# Content Based filtering
The content-based filtering approach takes advantage of the detailed textual features available for each article in the EB-NeRD dataset, including titles, abstracts, and full bodies. This method builds user profiles based on their individual reading history and recommends new articles that are similar in content to those previously read. Nevertheless, the approach may face challenges in promoting content diversity and in handling cold start scenarios for users with no interaction history.

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

Load the datasets

In [9]:
articles = pd.read_parquet("./articles.parquet")
history = pd.read_parquet("./train/history.parquet")
behaviors = pd.read_parquet("./train/behaviors.parquet")

This section prepares the textual data of the articles by combining the title, subtitle, and body. A TF-IDF vectorization is then applied using a list of Danish stopwords, in order to obtain a numerical representation of the articles suitable for content-based recommendation.

In [10]:
articles['text'] = (
    articles['title'].fillna('') + ' ' +
    articles['subtitle'].fillna('') + ' ' +
    articles['body'].fillna('')
)

article_text = articles['text']

article_to_index = pd.Series(articles.index, index=articles['article_id']).drop_duplicates()

article_titles = articles['title']

danish_stopwords = [
    "og", "i", "det", "er", "som", "på", "de", "en", "til", "med", "at", "for", "der", "af", "han"
]

tfidf = TfidfVectorizer(stop_words=danish_stopwords)

article_matrix = tfidf.fit_transform(article_text)

This function generates content-based article recommendations for a given user. It first retrieves the articles the user has previously viewed, then creates a user profile by averaging the TF-IDF vectors of those articles. It calculates the cosine similarity between the user profile and all article vectors to find the most similar ones. Finally, it returns the top N articles that the user hasn't seen yet.

In [11]:
def recommend_for_user(user_id, history, articles, article_matrix, article_to_index, top_n=5):
    user_id = int(user_id)  # Convertir l'ID de l'utilisateur en entier si nécessaire
    
    # Obtenir les articles que l'utilisateur a consultés
    user_articles = history[history['user_id'] == user_id]['article_id_fixed'].unique()
    
    # Convertir tous les article_id en int (pour comparaison fiable)
    user_articles = [int(aid) for aid in user_articles]
    
    # Obtenir les indices des articles que l'utilisateur a vus
    indices = [article_to_index[aid] for aid in user_articles if aid in article_to_index.index]
    
    if not indices:
        return []
    
    # Créer un profil utilisateur en moyennant les vecteurs des articles vus
    user_profile = article_matrix[indices].mean(axis=0)

    # Convertir en numpy array pour éviter l'erreur de type np.matrix
    user_profile = np.asarray(user_profile)

    # Calculer les similarités avec tous les articles
    sim_scores = cosine_similarity(user_profile.reshape(1, -1), article_matrix).flatten()

    # Écarter les articles déjà vus
    already_seen = set(indices)
    
    # Trier les similarités et choisir les top_n recommandations
    top_indices = np.argsort(sim_scores)[::-1]
    recommendations = []

    for idx in top_indices:
        if idx not in already_seen:
            recommendations.append(articles.iloc[idx])
            if len(recommendations) == top_n:
                break
    
    return recommendations

Example

In [22]:
# I need to transform evey numpy arrays in severals lignes
history_exploded = history.explode('article_id_fixed').copy()

user_id = '581228'  
recommendations = recommend_for_user(user_id, history_exploded, articles, article_matrix, article_to_index, top_n=5)

# Print the recommandation
for i, rec in enumerate(recommendations, 1):
    print(f"Recommandation {i}:")
    print(f"{rec['title']}") 
    print("--------------")

Recommandation 1:
Knivstikkeri i Fælledparken: En anholdt og sigtet
--------------
Recommandation 2:
City er klar til hævn over Real Madrid i CL-semifinale
--------------
Recommandation 3:
Drømmescoringer i gigant-brag
--------------
Recommandation 4:
Michael Laudrup: - Jeg synes altså kæden falder helt af
--------------
Recommandation 5:
Fejringen endte i kaos
--------------
